In [73]:
from functions import clean_data, split_data, build_tree, predict, tune_tree
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

In [66]:
X, Y = clean_data("claims_train.csv")

X_train, X_val, y_train, y_val = split_data(X, Y, 0.2)

In [ ]:
X_test, y_test = clean_data("claims_test.csv")

In [ ]:
y_train = y_train.to_numpy()

In [ ]:
best_params, best_mse = tune_tree(
    X_train, y_train,
    max_depth_list=[1, 2, 3, 4],
    min_samples_split_list=[2, 4, 6, 8],
    k=3
)

max_depth_final = best_params[0]
min_samples_split_final = best_params[1]

In [ ]:
# max_depth_final = 
# min_samples_split_final = 

In [61]:
final_tree = build_tree(
    X_train, y_train,
    max_depth=max_depth_final,
    min_samples_split=min_samples_split_final
)

In [ ]:
training_predictions = predict(X_train, final_tree)
validation_predictions = predict(X_val, final_tree)
test_predictions = predict(X_test, final_tree)

train_mse = np.mean((y_train - training_predictions) ** 2)
val_mse = np.mean((y_val - validation_predictions) ** 2)
test_mse = np.mean((y_test - test_predictions) ** 2)
print("Training MSE in model from scratch:", train_mse)
print("Validation MSE in model from scratch:", val_mse)
print("Validation MSE in model from scratch:", test_mse)

Train MSE: 0.05707582549650598
Validation MSE: 0.05557070119316541


In [ ]:
cv = KFold(n_splits=3, shuffle=True, random_state=1)

dt_regressor = DecisionTreeRegressor(random_state=1)
dt_params = {'max_depth': [1, 2, 3, 4],
             'min_samples_split': [2, 4, 6, 8]}

dt = GridSearchCV(
    dt_regressor, 
    param_grid=dt_params, 
    cv=cv, 
    scoring='neg_mean_squared_error', 
    n_jobs=-1 
)
dt.fit(X_train, y_train)

dt_best = dt.best_estimator_

print(dt_best)

In [ ]:
training_predictions_skl = dt_best.predict(X_train)
validation_predictions_skl = dt_best.predict(X_val)
test_predictions_skl = dt_best.predict(X_test)

train_mse_skl = mean_squared_error(y_train, training_predictions_skl)
val_mse_skl = mean_squared_error(y_val, validation_predictions_skl)
test_mse_skl = mean_squared_error(y_val, test_predictions_skl)

print("Train MSE sklearn:", train_mse_skl)
print("Validation MSE sklearn:", val_mse_skl)
print("Test MSE sklearn:", test_mse_skl)

Testing correctness

In [67]:
y_train = y_train.to_numpy()

In [68]:
subset_size = 4000  

subset_indices = np.random.choice(len(X_train), size=subset_size, replace=False)
X_train_subset = X_train[subset_indices]
y_train_subset = y_train[subset_indices]

In [69]:
best_params, best_mse = tune_tree(
    X_train_subset, y_train_subset,
    max_depth_list=[1, 2, 3, 4],
    min_samples_split_list=[2, 4, 6, 8],
    k=3
)

depth=1, split=2 -> avg MSE: 0.0656
depth=1, split=4 -> avg MSE: 0.0651
depth=1, split=6 -> avg MSE: 0.0653
depth=1, split=8 -> avg MSE: 0.0649
depth=2, split=2 -> avg MSE: 0.0655
depth=2, split=4 -> avg MSE: 0.0675
depth=2, split=6 -> avg MSE: 0.0675
depth=2, split=8 -> avg MSE: 0.0654
depth=3, split=2 -> avg MSE: 0.0683
depth=3, split=4 -> avg MSE: 0.0698
depth=3, split=6 -> avg MSE: 0.0666
depth=3, split=8 -> avg MSE: 0.0665
depth=4, split=2 -> avg MSE: 0.0683
depth=4, split=4 -> avg MSE: 0.0690
depth=4, split=6 -> avg MSE: 0.0706
depth=4, split=8 -> avg MSE: 0.0683

Best parameters: (1, 8)
Best CV average MSE: 0.06490832187428562


In [70]:
cv = KFold(n_splits=3, shuffle=True, random_state=1)

dt_regressor = DecisionTreeRegressor(random_state=1)
dt_params = {'max_depth': [1, 2, 3, 4],
             'min_samples_split': [2, 4, 6, 8]}

dt = GridSearchCV(
    dt_regressor, 
    param_grid=dt_params, 
    cv=cv, 
    scoring='neg_mean_squared_error', 
    n_jobs=-1 
)
dt.fit(X_train_subset, y_train_subset)

dt_best = dt.best_estimator_


In [74]:
train_preds_skl = dt_best.predict(X_train)
val_preds_skl = dt_best.predict(X_val)

train_mse_skl = mean_squared_error(y_train, train_preds_skl)
val_mse_skl = mean_squared_error(y_val, val_preds_skl)

print("\nSklearn Tree:")
print("Train MSE:", train_mse_skl)
print("Validation MSE:", val_mse_skl)


Sklearn Tree:
Train MSE: 0.05753236293873642
Validation MSE: 0.05594951093555061


In [71]:
print(dt_best)

DecisionTreeRegressor(max_depth=1, random_state=1)


In [72]:
dt.best_params_


{'max_depth': 1, 'min_samples_split': 2}

In [ ]:
# checking 
best_params_1, best_mse_1 = tune_tree(
    X_train_subset, y_train_subset,
    max_depth_list=[2,3],
    min_samples_split_list=[2,3],
    k=2
)

tree_1 = build_tree(
    X_train_subset, y_train_subset,
    max_depth=best_params_1[0],
    min_samples_split=best_params_1[1]
)

best_params_2, best_mse_2 = tune_tree(
    X_train_subset, y_train_subset,
    max_depth_list=[2,3],
    min_samples_split_list=[2,3],
    k=2
)

tree_2 = build_tree(
    X_train_subset, y_train_subset,
    max_depth=best_params_2[0],
    min_samples_split=best_params_2[1]
)

import json
json.dumps(tree_2, sort_keys=True) == json.dumps(tree_1, sort_keys=True)


depth=2, split=2 -> avg MSE: 0.0400
depth=2, split=3 -> avg MSE: 0.0600
depth=3, split=2 -> avg MSE: 0.0200
depth=3, split=3 -> avg MSE: 0.0400

Best parameters: (3, 2)
Best CV average MSE: 0.02
depth=2, split=2 -> avg MSE: 0.0400
depth=2, split=3 -> avg MSE: 0.0600
depth=3, split=2 -> avg MSE: 0.0200
depth=3, split=3 -> avg MSE: 0.0400

Best parameters: (3, 2)
Best CV average MSE: 0.02


True

In [ ]:
# best_params, best_mse = tune_tree(
#     X_train, y_train,
#     max_depth_list=[1, 2, 3, 4],
#     min_samples_split_list=[1, 2, 3, 4, 5],
#     # min_samples_split_list=[20, 50, 100, 1000],
#     k=3
# )

In [ ]:
X = np.array([[1], [2], [10], [12]])
y = np.array([1, 1, 5, 5])
tree = build_tree(X, y, max_depth=3, min_samples_split=1)
print(tree)
preds = predict(X, tree)
print(preds)
threshold = 10
left = y[X[:,0] < threshold]
right = y[X[:,0] >= threshold]
print(left, right)
print(np.sum((left - left.mean())**2) + np.sum((right - right.mean())**2))


{'feature': 0, 'threshold': np.int64(10), 'left': np.float64(1.0), 'right': np.float64(5.0)}
[1. 1. 5. 5.]
[1 1] [5 5]
0.0


In [ ]:
X = np.array([[1], [2], [10], [12]])
y = np.array([8, 8, 8, 8])
tree = build_tree(X, y, max_depth=3, min_samples_split=1)
print(tree)
preds = predict(X, tree)
print(preds)
threshold = 10
left = y[X[:,0] < threshold]
right = y[X[:,0] >= threshold]
print(left, right)
print(np.sum((left - left.mean())**2) + np.sum((right - right.mean())**2))

8.0
[8. 8. 8. 8.]
[8 8] [8 8]
0.0


In [ ]:
X = np.array([[1], [2], [10], [12]])
y = np.array([1, 1, 5, 5])
tree = build_tree(X, y, min_samples_split=5)
print(tree)

3.0


In [ ]:
tree = build_tree(X, y, max_depth=1)
print(tree)


{'feature': 0, 'threshold': np.int64(10), 'left': np.float64(1.0), 'right': np.float64(5.0)}
